# NYC Taxi Dataset Spatial Joins with H3 Turbo

This notebook demonstrates how to create a dataset resembling NYC Taxi pickups, convert the coordinates to H3 cells using the high-performance `h3_turbo` library, and perform rapid spatial joins against target zones. It also includes CPU comparisons.

In [37]:
import os
import sys
import glob

# Ensure h3_turbo is available in the environment
venv_patterns = [
    '/opt/venv/lib/python*/site-packages',
    '/app/.venv*/lib/python*/site-packages',
    '/workspace/.venv*/lib/python*/site-packages',
    os.path.join(os.getcwd(), '.venv312', 'lib', 'python*', 'site-packages')
]
found_paths = []
for pattern in venv_patterns:
    found_paths.extend(glob.glob(pattern))

for p in found_paths:
    if p not in sys.path:
        sys.path.append(p)

import h3_turbo
import numpy as np
import pandas as pd
import time
import h3

print('Libraries imported successfully.')

Libraries imported successfully.


## 1. Generate NYC Taxi Style Table
We generate 50 million random coordinates corresponding roughly to the bounding box of NYC.

In [38]:
def get_nyc_taxi_data(n_points=50_000_000):
    """
    Generate dummy NYC taxi dataset.
    NYC boundaries roughly: Lat 40.5 to 40.9, Lng -74.25 to -73.7
    """
    print(f"Generating {n_points:,} NYC taxi pickup points...")
    lats = np.random.uniform(40.5, 40.9, n_points).astype(np.float64)
    lngs = np.random.uniform(-74.25, -73.7, n_points).astype(np.float64)
    return lats, lngs

n_points = 50_000_000
lats, lngs = get_nyc_taxi_data(n_points)

# Display as a pandas DataFrame to represent a table
df = pd.DataFrame({'pickup_latitude': lats, 'pickup_longitude': lngs})
print("\nSample of the NYC Taxi table:")
display(df.head())

Generating 50,000,000 NYC taxi pickup points...

Sample of the NYC Taxi table:


,pickup_latitude,pickup_longitude
0,40.536049,-73.910715
1,40.794043,-74.052416
2,40.631532,-74.107476
3,40.758588,-74.131102
4,40.819265,-74.056802


## 2. Fast Lat/Lng to H3 Conversion (GPU/Accelerator)
We use `h3_turbo.latlons_to_h3s` to convert the 50 million latitude and longitude coordinates to H3 cells at resolution 9.

In [39]:
res = 9  # H3 Resolution for taxi zones (~174 meters edge length)

print(f"Converting {n_points:,} Lat/Lng pairs to H3 cells at resolution {res} using h3_turbo...")

# Warmup the JIT compiler for accurate benchmarking
h3_turbo.warmup()

start_time = time.time()
# High-performance batch conversion on GPU/Accelerators
h3_cells = h3_turbo.latlons_to_h3s(lats, lngs, res)
end_time = time.time()

turbo_elapsed = end_time - start_time
print(f"h3_turbo Conversion took {turbo_elapsed:.4f} seconds.")
print(f"Throughput: {n_points / turbo_elapsed:,.0f} conversions/second.")
print(f"Sample converted H3 cells (Hex): {[hex(x) for x in h3_cells[:5]]}")

# Add H3 cells to the table (stored as uint64)
df['pickup_h3'] = h3_cells

Converting 50,000,000 Lat/Lng pairs to H3 cells at resolution 9 using h3_turbo...
h3_turbo Conversion took 4.1169 seconds.
Throughput: 12,145,010 conversions/second.
Sample converted H3 cells (Hex): ['0x892a102b677ffff', '0x892a1009ba3ffff', '0x892a10625cfffff', '0x892a1073357ffff', '0x892a10098cfffff']


### 2.1 CPU Comparison: Lat/Lng to H3 Conversion
Let's benchmark the standard `h3-py` library running on the CPU for the same task. Note: We run this on the full 50 million dataset, which might take around 10-20 seconds.

In [40]:
print(f"Converting {n_points:,} Lat/Lng pairs to H3 cells at resolution {res} using h3-py (CPU)...")

start_time = time.time()
# CPU conversion (usually done via list comprehension or apply as h3-py is not vectorized natively)
cpu_h3_cells = [h3.latlng_to_cell(lat, lng, res) for lat, lng in zip(lats, lngs)]
end_time = time.time()

cpu_elapsed = end_time - start_time
print(f"CPU Conversion took {cpu_elapsed:.4f} seconds.")
print(f"CPU Throughput: {n_points / cpu_elapsed:,.0f} conversions/second.")
print(f"\nSPEEDUP: h3_turbo is {cpu_elapsed / turbo_elapsed:.2f}x faster than standard h3-py for conversions.")

Converting 50,000,000 Lat/Lng pairs to H3 cells at resolution 9 using h3-py (CPU)...
CPU Conversion took 102.2852 seconds.
CPU Throughput: 488,829 conversions/second.

SPEEDUP: h3_turbo is 24.85x faster than standard h3-py for conversions.


## 3. Generate Target Zones
We generate a set of target zones in the same area to simulate specific neighborhoods or high-demand taxi zones.

In [41]:
n_zones = 5_000
print(f"Generating {n_zones:,} target zones...")
zone_lats = np.random.uniform(40.5, 40.9, n_zones).astype(np.float64)
zone_lngs = np.random.uniform(-74.25, -73.7, n_zones).astype(np.float64)

zone_cells = h3_turbo.latlons_to_h3s(zone_lats, zone_lngs, res)
# Ensure zones are unique for the spatial join
zone_cells = np.unique(zone_cells)

print(f"Generated {len(zone_cells):,} unique target H3 zones.")

Generating 5,000 target zones...
Generated 4,400 unique target H3 zones.


## 4. Perform Spatial Join (GPU/Accelerator)
We use `h3_turbo.spatial_join` to test which of our 50 million pickups fall inside any of our target zones. This simulates a high-throughput Point-in-Polygon spatial join operation.

In [42]:
print(f"Performing Spatial Join using h3_turbo: {n_points:,} pickups against {len(zone_cells):,} zones...")

start_time = time.time()
# Perform high-speed spatial join
# Returns a boolean mask (or uint8 mask) of points that fall within any of the provided zones
is_in_zone = h3_turbo.spatial_join(h3_cells, zone_cells, res)
end_time = time.time()

turbo_join_elapsed = end_time - start_time
print(f"h3_turbo Spatial join took {turbo_join_elapsed:.4f} seconds.")
print(f"Throughput: {n_points / turbo_join_elapsed:,.0f} points/second.")

points_in_zones = np.sum(is_in_zone)
print(f"Found {points_in_zones:,} points inside the target zones.")
print(f"Percentage of pickups in target zones: {(points_in_zones / n_points) * 100:.2f}%")

# Add the result mask to our table
df['in_target_zone'] = is_in_zone
print("\nFinal Dataset with Spatial Join Results:")
display(df.head())

Performing Spatial Join using h3_turbo: 50,000,000 pickups against 4,400 zones...
h3_turbo Spatial join took 0.1176 seconds.
Throughput: 425,117,624 points/second.
Found 11,229,492 points inside the target zones.
Percentage of pickups in target zones: 22.46%

Final Dataset with Spatial Join Results:


,pickup_latitude,pickup_longitude,pickup_h3,in_target_zone
0,40.536049,-73.910715,617733131926503423,0
1,40.794043,-74.052416,617733122886467583,0
2,40.631532,-74.107476,617733146679443455,0
3,40.758588,-74.131102,617733151201427455,0
4,40.819265,-74.056802,617733122839019519,1


### 4.1 CPU Comparison: Spatial Join
Let's benchmark a standard CPU approach. Since the H3 cells are already computed, the fastest CPU approach in Python is typically using NumPy's `isin` function (or a set lookup).

In [43]:
print(f"Performing Spatial Join using NumPy (CPU): {n_points:,} pickups against {len(zone_cells):,} zones...")

start_time = time.time()
# Fastest CPU vector approach in Python: NumPy isin
cpu_is_in_zone = np.isin(h3_cells, zone_cells)
end_time = time.time()

cpu_join_elapsed = end_time - start_time
print(f"CPU Spatial join took {cpu_join_elapsed:.4f} seconds.")
print(f"CPU Throughput: {n_points / cpu_join_elapsed:,.0f} points/second.")
print(f"\nSPEEDUP: h3_turbo is {cpu_join_elapsed / turbo_join_elapsed:.2f}x faster than NumPy CPU execution for spatial joins.")

Performing Spatial Join using NumPy (CPU): 50,000,000 pickups against 4,400 zones...
CPU Spatial join took 6.1350 seconds.
CPU Throughput: 8,150,023 points/second.

SPEEDUP: h3_turbo is 52.16x faster than NumPy CPU execution for spatial joins.
